In [1]:
# ============================================================
# Notebook 3: 03_metrics_bootstrap_dei.ipynb
# Purpose : From saved recourse, compute violation rates,
#           Reliable RR, bootstrap & exact CIs, and the
#           Double Exclusion Index (DEI). Light & re-runnable.
# ============================================================

import os, pickle
import numpy as np
import pandas as pd
from scipy.stats import beta

# ------------------------------------------------------------
# 0. Load artifacts from Notebooks 1 & 2
# ------------------------------------------------------------
DATA_DIR = r"C:\Users\miy\Downloads\1000_논문투고\117.금융 합성데이터\3.개방데이터\1.데이터\1. 합성데이터"
ARTIFACT_DIR = os.path.join(DATA_DIR, "_artifacts")
with open(os.path.join(ARTIFACT_DIR, "nb1_artifacts.pkl"), "rb") as f:
    A = pickle.load(f)
with open(os.path.join(ARTIFACT_DIR, "recourse_main.pkl"), "rb") as f:
    recourse = pickle.load(f)
with open(os.path.join(ARTIFACT_DIR, "recourse_dis_full.pkl"), "rb") as f:
    recourse_dis_full = pickle.load(f)
with open(os.path.join(ARTIFACT_DIR, "nb2_frames.pkl"), "rb") as f:
    frames = pickle.load(f)

FEATURES   = A["FEATURES"]; IMMUTABLE = A["IMMUTABLE"]; ACTIONABLE = A["ACTIONABLE"]
cdr_table  = A["cdr_table"]
rejected   = frames["rejected"]; dis_full = frames["dis_full"]
ACTION_HORIZON = 0.20

# ------------------------------------------------------------
# 1. Violation checks (shared helpers)
# ------------------------------------------------------------
def check_path(orig_row, cf_row):
    imm_viol = any(not np.isclose(float(orig_row[f]), float(cf_row[f])) for f in IMMUTABLE)
    act_viol = False
    for f in ACTIONABLE:
        o = float(orig_row[f]); c = float(cf_row[f])
        act_viol |= (c != 0) if o == 0 else abs(c - o)/abs(o) > ACTION_HORIZON + 1e-9
    return imm_viol, act_viol

def borrower_flags(sample_df, cf_list):
    found, reliable = [], []
    for (_, orig), cfs in zip(sample_df.iterrows(), cf_list):
        if cfs is None or len(cfs) == 0:
            found.append(0); reliable.append(0); continue
        found.append(1); rel = False
        for _, cf in cfs.iterrows():
            iv, av = check_path(orig, cf)
            if not iv and not av: rel = True; break
        reliable.append(1 if rel else 0)
    return np.array(found), np.array(reliable)

def score_scenario(sample_df, cf_list):
    n = len(sample_df); rr = rel_b = pt = pr = 0
    for (_, orig), cfs in zip(sample_df.iterrows(), cf_list):
        if cfs is None or len(cfs) == 0: continue
        rr += 1; b_rel = False
        for _, cf in cfs.iterrows():
            pt += 1; iv, av = check_path(orig, cf)
            if not iv and not av: pr += 1; b_rel = True
        if b_rel: rel_b += 1
    return {"RR": rr/n, "Borrower_Reliable_RR": rel_b/n,
            "Pathway_Reliable_RR": (pr/pt if pt else 0.0), "n": n}

# ------------------------------------------------------------
# 2. Main metrics table (sampled groups, 3 scenarios)
# ------------------------------------------------------------
rows = []
for gname in ["advantaged", "disadvantaged"]:
    s = recourse[gname]["_sample"]
    for scen in ["A", "B", "C"]:
        r = score_scenario(s, recourse[gname][scen])
        r.update({"group": gname, "scenario": scen}); rows.append(r)
res = pd.DataFrame(rows)[["group","scenario","n","RR","Pathway_Reliable_RR","Borrower_Reliable_RR"]]
for c in ["RR","Pathway_Reliable_RR","Borrower_Reliable_RR"]:
    res[c] = (res[c]*100).round(2)
print("=== Main metrics (n=200 per group) ===")
print(res.to_string(index=False))

# ------------------------------------------------------------
# 3. Bootstrap 95% CIs (percentile) for RR & Borrower Reliable RR
# ------------------------------------------------------------
rng = np.random.default_rng(42); N_BOOT = 2000
def boot_ci(flags):
    n = len(flags); means = np.array([flags[rng.integers(0,n,n)].mean() for _ in range(N_BOOT)])
    return flags.mean(), *np.percentile(means, [2.5, 97.5])

ci_rows = []
for gname in ["advantaged", "disadvantaged"]:
    s = recourse[gname]["_sample"]
    for scen in ["A","B","C"]:
        found, reliable = borrower_flags(s, recourse[gname][scen])
        for mname, arr in [("RR", found), ("Borrower_Reliable_RR", reliable)]:
            m, lo, hi = boot_ci(arr)
            ci_rows.append({"group":gname,"scenario":scen,"metric":mname,
                            "est_%":round(m*100,2),"lo_%":round(lo*100,2),"hi_%":round(hi*100,2)})
print("\n=== Bootstrap 95% CIs ===")
print(pd.DataFrame(ci_rows).to_string(index=False))

# ------------------------------------------------------------
# 4. Full-population disadvantaged: realized metrics + exact CI
# ------------------------------------------------------------
def cp_upper(k, n, alpha=0.05):
    return 1.0 if k == n else beta.ppf(1-alpha/2, k+1, n-k)

full_rows = []
for scen in ["A","B","C"]:
    found, reliable = borrower_flags(dis_full, recourse_dis_full[scen])
    n = len(dis_full); k = int(reliable.sum())
    full_rows.append({"scenario":scen,"n":n,"RR_%":round(100*found.mean(),3),
                      "n_reliable":k,"Reliable_RR_%":round(100*k/n,3),
                      "Reliable_upper95_%":round(100*cp_upper(k,n),3)})
print("\n=== Full disadvantaged pool (exact) ===")
print(pd.DataFrame(full_rows).to_string(index=False))

# ------------------------------------------------------------
# 5. Double Exclusion Index (Method A: conditional probability)
# ------------------------------------------------------------
p_vip = {"advantaged": cdr_table["advantaged"]/100,
         "disadvantaged": cdr_table["disadvantaged"]/100}
relC = (res[res.scenario=="C"].set_index("group")["Borrower_Reliable_RR"]/100).to_dict()

dei_rows = []
for g in ["advantaged","disadvantaged"]:
    p_reject = 1 - p_vip[g]; p_no_rec = 1 - relC.get(g, 0.0)
    dei_rows.append({"group":g,"P_reject":round(p_reject,4),
                     "P_no_reliable_recourse":round(p_no_rec,4),
                     "DEI":round(p_reject*p_no_rec,4)})
dei_df = pd.DataFrame(dei_rows)
print("\n=== Double Exclusion Index (Method A) ===")
print(dei_df.to_string(index=False))
ratio = dei_df.set_index("group").loc["disadvantaged","DEI"] / \
        dei_df.set_index("group").loc["advantaged","DEI"]
print(f"DEI ratio (disadvantaged/advantaged): {ratio:.2f}x")

=== Main metrics (n=200 per group) ===
        group scenario   n    RR  Pathway_Reliable_RR  Borrower_Reliable_RR
   advantaged        A 200 100.0                 2.00                   6.5
   advantaged        B 200 100.0                 1.88                   5.5
   advantaged        C 200  15.0                99.07                  15.0
disadvantaged        A 200 100.0                 0.00                   0.0
disadvantaged        B 200 100.0                 0.00                   0.0
disadvantaged        C 200   0.0                 0.00                   0.0

=== Bootstrap 95% CIs ===
        group scenario               metric  est_%  lo_%  hi_%
   advantaged        A                   RR  100.0 100.0 100.0
   advantaged        A Borrower_Reliable_RR    6.5   3.5  10.0
   advantaged        B                   RR  100.0 100.0 100.0
   advantaged        B Borrower_Reliable_RR    5.5   2.5   9.0
   advantaged        C                   RR   15.0  10.5  20.0
   advantaged        C B